# **Install necessary libraries**

In [ ]:
# Unzip the library
!unzip unsloth.zip

# Go insiade the directory
%cd /content/unsloth

# Install unsloth library with the configuration
!pip install -e ".[cu128-torch280]"

# Install ML experiments tracker
!pip pip install wandb

# Install flash-attention-2
!pip install flash-attn --no-build-isolation \
    --find-links https://github.com/Dao-AILab/flash-attention/releases/tag/v2.8.3

In [ ]:
!pip install --upgrade --no-cache-dir unsloth vllm trl transformers

# **Initialize Weights and Biases (WANDB)**

In [ ]:
import wandb
import os
from google.colab import userdata

# Inject api key & project to `os.environ`
os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = userdata.get("WANDB_PROJECT")

if wandb.login():
    print("Successfully logged !")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tahar-masmaliyev (tahar-masmaliyev-george-washington-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged !


# **Load Libraries**

In [ ]:
import torch
import re
from unsloth import FastLanguageModel

from trl import GRPOConfig, GRPOTrainer

from datasets import (load_dataset, concatenate_datasets,
                      Dataset, DatasetDict)

from dotenv import load_dotenv

import os
from tqdm import tqdm
from typing import Dict, List, Optional, Any


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Initialize Model**

In [ ]:
# Configure model settings
model_name = "taharmasmaliyev07/Qwen-3-4B-b16-tuned-full"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 16
MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH

In [ ]:
# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
)

# Apply PEFT (LORA) wrapper
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# **Dataset Loader & Tokenizer**

In [ ]:
def tokenize(example: Dict[str, str]) -> Dict[str, List[int]]:
    question = example['question']

    prompt_message = [
        {'role': 'user',      'content': question}
    ]

    input_ids = tokenizer.apply_chat_template(
        prompt_message,
        tokenize=True,
        add_generation_prompt=True,
        truncation=True,
        max_length=MAX_PROMPT_LENGTH
    )

    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids)
    }

def prepare_grpo_dataset(ds: Dataset) -> Dataset:
    def process(example):
        return {
            'prompt': [
                {'role': 'user', 'content': example['question']}
            ],
            'answer': example['answer']
        }

    return ds.map(process, remove_columns=ds.column_names)

In [ ]:
# Load the train& test datasets
ds_train = load_dataset('taharmasmaliyev07/arm-rl-stage-dataset', split='train')

dsp_train = prepare_grpo_dataset(ds_train)

README.md:   0%|          | 0.00/413 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/828k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7540 [00:00<?, ? examples/s]

Map:   0%|          | 0/19800 [00:00<?, ? examples/s]

# **Reward Function**

In [ ]:
def extract_answer(text):
    match = re.search(r'<ANSWER>\s*(.+?)\s*</ANSWER>', text)
    return match.group(1) if match else ""

def correctness_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    print(prompts)
    print(completions)
    print(answer)

    for completion, gold in zip(completions, answer):
        # Extract text from chat format
        response = completion[0]['content']

        gold_str = str(gold).strip()
        ans = extract_answer(response)

        if ans == gold_str:
            rewards.append(1.0)
        else:
            rewards.append(-1.0)

    return rewards

# **Model training**

In [ ]:
# Initialize training configuration
training_args = GRPOConfig(
    # Batch & generation settings
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 4,
    num_generations = 4,

    # Training duration
    num_train_epochs = 2,
    warmup_ratio = 0.1,

    # Learning rate & optimizer
    optim = 'adamw_8bit',
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,

    # Logging
    logging_steps = 25,

    lr_scheduler_type = 'cosine',
    load_best_model_at_end = True,

    # Sequence lengths
    max_prompt_length = MAX_PROMPT_LENGTH,
    max_completion_length = MAX_COMPLETION_LENGTH,

    # Precision
    bf16 = True,

    # Reporting
    report_to = 'wandb',
    run_name=f"{model_name}-b16-r{LORA_RANK}-RL",
    disable_tqdm = False
)

# Initialize Trainer with reward functions
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    args = training_args,
    train_dataset = dsp_train,
    reward_funcs = [
        correctness_reward_func,    # +1 correct, -1 wrong (main signal!)
        # length_penalty_func,        # small penalty for too short/long
    ],
)

trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,800 | Num Epochs = 2 | Total steps = 4,950
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 40960}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!
[[{'content': 'The diameter $AB$ of a circle of radius $2$ is extended to a point $D$ outside the circle so that $BD=3$. Point $E$ is chosen so that $ED=5$ and line $ED$ is perpendicular to line $AD$. Segment $AE$ intersects the circle at a point $C$ between $A$ and $E$. What is the area of $\\triangle  ABC$?\n$\\textbf{(A)}\\ \\frac{120}{37}\\qquad\\textbf{(B)}\\ \\frac{140}{39}\\qquad\\textbf{(C)}\\ \\frac{145}{39}\\qquad\\textbf{(D)}\\ \\frac{140}{37}\\qquad\\textbf{(E)}\\ \\frac{120}{31}$\n', 'role': 'user'}], [{'content': 'The diameter $AB$ of a circle of radius $2$ is extended to a point $D$ outside the circle so that $BD=3$. Point $E$ is chosen so that $ED=5$ and line $ED$ is perpendicular to line $AD$. Segment $AE$ intersects the circle at a point $C$ between $A$ and $E$. What is the area of $\\triangle  ABC$?\n$\\textbf{(A)}\\ \\frac{120}{37}\\qquad\\textbf{(B)}\\ \\frac{140}{39}\\qquad\\textbf{(C)}\\ \\frac{145}{39}\\qquad

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std


[[{'content': 'Gina chooses what she and her sister will watch on Netflix three times as often as her sister does. If her sister watches a total of 24 shows on Netflix per week, and each show is 50 minutes long, how many minutes of Netflix does Gina get to choose?', 'role': 'user'}], [{'content': 'Gina chooses what she and her sister will watch on Netflix three times as often as her sister does. If her sister watches a total of 24 shows on Netflix per week, and each show is 50 minutes long, how many minutes of Netflix does Gina get to choose?', 'role': 'user'}], [{'content': 'Gina chooses what she and her sister will watch on Netflix three times as often as her sister does. If her sister watches a total of 24 shows on Netflix per week, and each show is 50 minutes long, how many minutes of Netflix does Gina get to choose?', 'role': 'user'}], [{'content': 'Gina chooses what she and her sister will watch on Netflix three times as often as her sister does. If her sister watches a total of 

KeyboardInterrupt: 